In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/khakim/shakespeare/input.txt


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cuda


In [4]:
class FeedForward(nn.Module):
    
    def __init__(self, embed_size, dropout = 0.1):
        super().__init__()
        self.lin1 = nn.Linear(embed_size, embed_size * 4)
        self.act = nn.SiLU()
        self.lin2 = nn.Linear(embed_size * 4, embed_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # (batch_size, block_size, embed_size)
        return self.dropout(self.lin2(self.act(self.lin1(x))))

class Block(nn.Module):
    
    def __init__(self, block_size, embed_size, num_head, dropout = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_size)
        self.attn = nn.MultiheadAttention(embed_size, num_head, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_size)
        self.ff = FeedForward(embed_size, dropout = dropout)
        self.dropout1 = nn.Dropout(dropout)
        mask = torch.triu(torch.ones(block_size, block_size), diagonal=1).bool()
        self.register_buffer('mask', mask)
    
    def forward(self, x):
        # (batch_size, block_size, embed_size)
        # adding residual connections!
        x = self.norm1(x)
        attMat = self.attn(key = x, query = x, value = x, attn_mask = self.mask[:x.shape[-2], :x.shape[-2]])[0]
        x = x + self.dropout1(attMat)
        x = x + self.ff(self.norm2(x))
        return x

class GPT(nn.Module):

    def __init__(self, block_size, alph_size, embed_size, num_head, num_blocks, dropout = 0.1):
        super().__init__()
        self.block_size = block_size
        self.embedding = nn.Embedding(alph_size, embed_size)
        self.pos_embedding = nn.Embedding(block_size, embed_size)
        self.dropout_embed = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block(block_size, embed_size, num_head, dropout) for _ in range(num_blocks)])
        self.norm = nn.LayerNorm(embed_size)
        self.linear = nn.Linear(embed_size, alph_size)
    
    def forward(self, x):
        # (batch_size, block_size), numbers from zero to alph_size - 1
        x_embed = self.embedding(x) # (batch_size, block_size, embed_size)
        x_pos_embed = self.pos_embedding(torch.arange(x.shape[-1], device = x.device)) # (block_size, embed_size)
        x_embed = self.dropout_embed(x_embed + x_pos_embed) # (batch_size, block_size, embed_size) + (block_size, embed_size) -> (batch_size, block_size, embed_size)
        logits = self.blocks(x_embed) # (batch_size, block_size, embed_size)
        logits = self.norm(logits) # (batch_size, block_size, embed_size)
        logits = self.linear(logits) # (batch_size, block_size, alph_size)
        return logits
    
    @torch.no_grad()
    def generate(self, context, gen_len):
        text = ''
        context = encode(context[-min(len(context), self.block_size):])
        for _ in range(gen_len):
            logits = self.__call__(torch.tensor([context], device = device))
            ix = torch.multinomial(F.softmax(logits[-1, -1], -1), num_samples=1).item()
            text += itos[ix]
            if len(context) == self.block_size:
                context = context[1:]
            context = context + [ix]
        return text



In [5]:
gen = torch.Generator().manual_seed(42)
embed_size = 192
num_head = 4
head_size = embed_size // num_head
num_blocks = 8
batch_size = 64
block_size = 384

In [6]:
text = open('/kaggle/input/datasets/khakim/shakespeare/input.txt').read()

alph =  sorted(list(set(text)))
alph_size = len(alph)
stoi = {c: i for i, c in enumerate(alph)}
itos = {v: k for k, v in stoi.items()}
encode = lambda x: [stoi[c] for c in x]
decode = lambda x: ''.join([itos[ic] for ic in x])
n = int(0.9 * len(text))
Xtr = encode(text[:n])
Xval = encode(text[n:])

@torch.no_grad()
def getbatch(X):
    starts = torch.randint(0, len(X) - block_size, (batch_size,), generator = gen)
    context = torch.tensor([X[st : st + block_size] for st in starts]).to(device)
    target = torch.tensor([X[st + 1 : st + block_size + 1] for st in starts]).to(device)
    return context, target

# model overview
model = GPT(block_size, alph_size, embed_size, num_head, num_blocks).to(device)
print(f'Total number of parameters: {sum(p.numel() for p in model.parameters())}')

# utils
@torch.no_grad()
def getloss(X, num_samples):
    sumloss = 0
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        for _ in range(num_samples):
            x, y = getbatch(X)
            logits = model(x)
            loss = F.cross_entropy(logits.view(batch_size * block_size, alph_size), y.view(batch_size * block_size))
            sumloss += loss
    return sumloss / num_samples

Total number of parameters: 3658049


In [7]:
params = list(model.parameters())
lr = 5e-4
patience = 3  # сколько eval-шагов ждать без улучшения
patience_counter = 0
optimizer = torch.optim.AdamW(params, lr)
epochs = 6000
eval_every = 500
best_val_loss = float('inf')
best_model_path = '/kaggle/working/best_gpt_classic.pth'
latest_checkpoint_path = '/kaggle/working/latest_checkpoint.pth'

losses = []
train_losses = []
val_losses = []
scaler = torch.amp.GradScaler()
for e in range(epochs + 1):
    x, y = getbatch(Xtr)
    model.train()
    optimizer.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        logits = model(x)
        loss = F.cross_entropy(logits.view(batch_size * block_size, alph_size), y.view(batch_size * block_size))
        losses.append(loss.item())
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    if e % eval_every == 0:
        model.eval()
        train_loss = getloss(Xtr, 50)
        val_loss = getloss(Xval, 50)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f'Epoch {e}/{epochs}: train loss {train_loss}, val loss {val_loss}')
        torch.save({
                'epoch': e,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, latest_checkpoint_path)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({
                'epoch': e,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, best_model_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {e}')
                break

torch.save({
            'train_losses': torch.tensor(train_losses),
            'val_losses': torch.tensor(val_losses),
            'all_losses': torch.tensor(losses)}, '/kaggle/working/loss_stats.pth')

Epoch 0/6000: train loss 3.5561788082122803, val loss 3.564185380935669
Epoch 500/6000: train loss 1.9826719760894775, val loss 2.0717363357543945
Epoch 1000/6000: train loss 1.5669394731521606, val loss 1.741373062133789
Epoch 1500/6000: train loss 1.4024312496185303, val loss 1.6189417839050293
Epoch 2000/6000: train loss 1.3113398551940918, val loss 1.5533798933029175
Epoch 2500/6000: train loss 1.248517394065857, val loss 1.5220625400543213
Epoch 3000/6000: train loss 1.1978868246078491, val loss 1.520874261856079
Epoch 3500/6000: train loss 1.151429533958435, val loss 1.5193456411361694
Epoch 4000/6000: train loss 1.1165516376495361, val loss 1.5163836479187012
Epoch 4500/6000: train loss 1.0731042623519897, val loss 1.5356740951538086
Epoch 5000/6000: train loss 1.038993000984192, val loss 1.5498346090316772
Epoch 5500/6000: train loss 0.9968903064727783, val loss 1.5686819553375244
Early stopping at epoch 5500


In [8]:
import os
print(os.listdir('/kaggle/working/'))

['.virtual_documents', 'loss_stats.pth', 'latest_checkpoint.pth', 'best_gpt_classic.pth']
